# 07 — Conforto Térmico (UTCI) e Fluxo de Calor Antropogênico

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ByMaxAnjos/LCZ4py/blob/main/notebooks/local/07_thermal_comfort_anthropogenic_heat.pt.ipynb)

**Objetivo de aprendizagem**: calcular o Índice Térmico Universal (UTCI) — o índice biometeorológico padronizado internacionalmente, usado em sistemas de alerta de saúde relacionados ao calor — e estimar o fluxo de calor antropogênico (Q_F, o calor residual liberado por edifícios, tráfego e pessoas), ambos determinados diretamente pela classe LCZ. Ao final, você entenderá *por que* dois bairros com a mesma temperatura do ar podem representar riscos de estresse térmico muito diferentes para as pessoas, e por que LCZs urbanas compactas também são produtoras líquidas de calor extra que a própria cidade precisa absorver.

## Por que a classe LCZ importa para o conforto térmico humano e o calor antropogênico

O esquema LCZ de Stewart & Oke (2012) foi construído em torno de propriedades físicas mensuráveis da forma urbana — fração de área construída, fração de superfície impermeável, razão altura/largura e o **fator de visão do céu (sky view factor, SVF)**, entre outras. Duas dessas propriedades alimentam diretamente perigos climáticos em escala humana que um simples mapa de temperatura do ar não consegue capturar sozinho.

**Conforto térmico e o UTCI.** O Índice Térmico Universal (Fiala et al. 2012, ISO 7243) é o índice em que a maioria dos sistemas de alerta precoce de saúde relacionados ao calor no mundo se baseia, pois combina as quatro variáveis que realmente determinam como o corpo humano percebe o calor: temperatura do ar, velocidade do vento, umidade e a **temperatura radiante média (Tmrt)** — a carga radiante líquida recebida de todas as superfícies ao redor (paredes, pavimento, céu). A temperatura do ar isolada é um indicador fraco de estresse térmico nas cidades: uma pessoa em um cânion urbano estreito e de baixo SVF, típico de uma zona Compact Highrise (LCZ 1), recebe muito mais radiação de onda longa refletida e reemitida pelas paredes e pavimento ao redor do que alguém em um campo aberto (LCZ D / classe 14), mesmo que um termômetro registre exatamente a mesma temperatura do ar nos dois locais. A função `lcz_utci` captura isso estimando a Tmrt a partir do fator de visão do céu da LCZ quando não há uma medição direta de Tmrt (rara fora de campanhas micrometeorológicas dedicadas) — LCZs compactas e de baixo SVF retêm mais radiação de onda longa e elevam a Tmrt (e, portanto, o UTCI) bem acima da temperatura do ar, enquanto LCZs abertas, de alto SVF, e coberturas naturais permanecem muito mais próximas dela. Esse é exatamente o mecanismo explorado quantitativamente no artigo do framework LCZ4r (Anjos et al. 2025, *Scientific Reports*, https://www.nature.com/articles/s41598-025-92000-0).

**Fluxo de calor antropogênico (Q_F).** O calor antropogênico é o calor residual liberado diretamente no balanço de energia urbano pela atividade humana — sistemas de aquecimento/resfriamento de edifícios, motores de veículos, processos industriais e o calor metabólico das próprias pessoas. É um verdadeiro *termo de fonte* no balanço de energia da superfície urbana, distinto (e adicional) aos efeitos de retenção de radiação e redução da evapotranspiração que tornam as cidades mais quentes. O Q_F se correlaciona fortemente com a classe LCZ porque densidade construtiva, impermeabilização e atividade de tráfego/industrial variam sistematicamente ao longo da tipologia LCZ — uma zona de Indústria Pesada (LCZ 10) ou um núcleo Compact Highrise produz rotineiramente uma ordem de grandeza a mais de calor residual por metro quadrado do que uma periferia Sparsely Built ou vegetada.

## Instalar o LCZ4py

In [1]:
!pip -q install "LCZ4py[all] @ git+https://github.com/ByMaxAnjos/LCZ4py.git"


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: /Applications/QGIS.app/Contents/MacOS/bin/python3 -m pip install --upgrade pip
ERROR: Package 'lcz4py' requires a different Python: 3.9.5 not in '>=3.10'


## Obter um mapa LCZ real para trabalhar

Usamos Vaduz (Liechtenstein) — uma cidade pequena cuja bounding box mantém o download rápido. Lemos o raster classificado como um array NumPy simples de códigos de classe LCZ (1–17, 0 = nodata) com `rasterio`; tanto `lcz_utci` quanto `lcz_anthropogenic_heat` são funções puramente NumPy que operam diretamente sobre esses arrays (ou sobre escalares/arrays 1-D), sem qualquer acesso adicional à rede. Também removemos os pixels de nodata (0) antes de qualquer estatística por classe — boa prática sempre que se agrega por classe LCZ, independentemente da função.

In [2]:
from LCZ4py import lcz_get_map
import rasterio
import numpy as np

map_path = lcz_get_map(city="Vaduz")
print(map_path)

with rasterio.open(map_path) as src:
    lcz_array = src.read(1).astype(np.int32)

print("Raster shape:", lcz_array.shape)
print("LCZ classes present:", np.unique(lcz_array))

# Drop nodata (0) pixels before feeding class values into lcz_anthropogenic_heat
# -- always mask nodata before per-class analysis, LCZ or otherwise.
lcz_valid = lcz_array[lcz_array > 0]
print("Valid (non-nodata) pixels:", lcz_valid.shape)

06:29:36 - LCZ4py._internal._lcz_map_engine - INFO - Loading clipped map from local cache.


/Users/co2map/.lcz4r_cache/clipped_a8db9faa23e32c3b.tif
Raster shape: (120, 131)
LCZ classes present: [ 0  5  6  8  9 11 12 14 15 17]
Valid (non-nodata) pixels: (2539,)


## Fluxo de calor antropogênico a partir de classes LCZ reais: `lcz_anthropogenic_heat`

`lcz_anthropogenic_heat(lcz_classes, method="simple"|"detailed", params=None, lang="en")` estima o Q_F (W/m²) para cada pixel de um array de classes LCZ e retorna um `AnthropogenicHeatResult` com:

- `array` — Q_F por pixel (W/m²), mesma forma do `lcz_classes` de entrada,
- `stats` — média/mín/máx/contagem de pixels por classe, além de uma média ponderada pela área (`total_mean`),
- `plot` — um heatmap Plotly (entrada 2-D) ou um gráfico de barras (entrada 1-D) do Q_F por classe.

**`method="simple"`** consulta uma faixa `(mín, máx)` de Q_F derivada da literatura para cada classe LCZ (por exemplo, 60–120 W/m² para Compact Highrise, 8–20 W/m² para Sparsely Built, 0–5 W/m² para todas as classes naturais/água) e atribui a cada pixel o ponto médio da faixa da sua classe. É rápido, transparente e fácil de sobrescrever via o dicionário `params`, mas ignora a variabilidade dentro da classe — todo pixel Compact Highrise recebe exatamente o mesmo valor, independentemente da densidade ou altura real daquele quarteirão específico.

**`method="detailed"`** deriva o Q_F a partir dos parâmetros morfológicos reais da LCZ (via a tabela de consulta de `lcz_get_parameters`): a fração de área construída (BSF) e a altura média dos edifícios (HRE) alimentam um termo de energia dos edifícios, a fração de superfície impermeável (ISF) alimenta um termo de tráfego, e uma proxy combinada de fração construída alimenta um termo metabólico humano simplificado. É mais fundamentado fisicamente (responde à geometria real de cada classe, em vez de um ponto médio fixo da literatura), mas ainda é uma parametrização simplificada, não um modelo de balanço de energia bottom-up calibrado — trate ambos os métodos como estimativas de ordem de grandeza, preferindo "detailed" quando você precisa de sensibilidade à geometria e "simple" quando só precisa de uma linha de base rápida e ancorada na literatura.

In [3]:
from LCZ4py.local.lcz_thermal import lcz_anthropogenic_heat

qf_simple = lcz_anthropogenic_heat(lcz_valid, method="simple")

print("Q_F array shape:", qf_simple.array.shape)
print("Area-weighted mean Q_F (simple):", round(qf_simple.stats["total_mean"], 2), "W/m2")
for cls, s in sorted(qf_simple.stats["per_class"].items()):
    print(f"  LCZ {cls:>2}: mean={s['mean']:.1f} W/m2  (n={s['count']} px)")

qf_simple.plot

Q_F array shape: (2539,)
Area-weighted mean Q_F (simple): 7.54 W/m2
  LCZ  5: mean=35.0 W/m2  (n=101 px)
  LCZ  6: mean=27.5 W/m2  (n=306 px)
  LCZ  8: mean=25.0 W/m2  (n=22 px)
  LCZ  9: mean=14.0 W/m2  (n=119 px)
  LCZ 11: mean=2.5 W/m2  (n=1260 px)
  LCZ 12: mean=2.5 W/m2  (n=88 px)
  LCZ 14: mean=2.5 W/m2  (n=244 px)
  LCZ 15: mean=2.5 W/m2  (n=389 px)
  LCZ 17: mean=2.5 W/m2  (n=10 px)


**Lendo a saída do `simple`**: `qf_simple.stats["total_mean"]` é o fluxo médio de calor residual, ponderado pela área, em todo o raster de Vaduz — uma pequena cidade alpina, portanto espere um valor modesto, dominado pelas classes residenciais de baixa densidade realmente presentes nessa área. O detalhamento por `per_class` mostra o Q_F de ponto médio fixo atribuído a cada classe encontrada no raster; todo pixel de uma dada classe recebe um valor idêntico aqui, por construção. O heatmap (ou gráfico de barras, se a entrada fosse 1-D) torna o contraste espacial entre cobertura construída e natural imediatamente visível.

In [4]:
qf_detailed = lcz_anthropogenic_heat(lcz_valid, method="detailed")

print("Area-weighted mean Q_F (detailed):", round(qf_detailed.stats["total_mean"], 2), "W/m2")
for cls, s in sorted(qf_detailed.stats["per_class"].items()):
    print(f"  LCZ {cls:>2}: mean={s['mean']:.1f} W/m2  (n={s['count']} px)")

qf_detailed.plot

Area-weighted mean Q_F (detailed): 7.99 W/m2
  LCZ  5: mean=30.8 W/m2  (n=101 px)
  LCZ  6: mean=14.2 W/m2  (n=306 px)
  LCZ  8: mean=19.0 W/m2  (n=22 px)
  LCZ  9: mean=7.1 W/m2  (n=119 px)
  LCZ 11: mean=8.3 W/m2  (n=1260 px)
  LCZ 12: mean=4.9 W/m2  (n=88 px)
  LCZ 14: mean=1.3 W/m2  (n=244 px)
  LCZ 15: mean=1.0 W/m2  (n=389 px)
  LCZ 17: mean=0.9 W/m2  (n=10 px)


**Lendo a saída do `detailed`**: compare `qf_detailed.stats["total_mean"]` e as médias por classe com a execução `simple` acima. Como o `detailed` é orientado pelos parâmetros reais de BSF/ISF/altura média de cada classe, em vez de um ponto médio fixo da literatura, os dois métodos podem divergir — especialmente para classes cuja geometria real está longe dos valores "típicos" que a faixa da literatura assume. Nenhum dos métodos é uma medição de referência (ground-truth); use `simple` para uma linha de base rápida ancorada na literatura e `detailed` quando quiser que o Q_F responda à morfologia específica do mapa LCZ que você está analisando (por exemplo, comparando duas cidades com o mesmo rótulo LCZ mas densidade construtiva real diferente).

## Estresse térmico a partir de um cenário sintético de onda de calor: `lcz_utci`

`lcz_utci(air_temp, wind_speed, relative_humidity, mean_radiant_temp=None, *, lc_z=None, output="category"|"index", lang="en")` calcula o UTCI a partir das quatro entradas meteorológicas e retorna um `UTCIResult` com:

- `values` — o índice UTCI numérico em °C-equivalente (sempre retornado como um array 1-D achatado, mesmo para uma entrada `lc_z` 2-D — remonte com `.reshape(lc_z.shape)` se precisar de um mapa espacial),
- `categories` — o rótulo da categoria de estresse térmico para cada valor (mesma forma de `values`),
- `stats` — média/mín/máx/desvio-padrão do índice, além do percentual de pixels em cada categoria,
- `plot` — uma figura Plotly (histograma, já que `values` é sempre 1-D no momento em que é plotado).

Se `mean_radiant_temp` não for fornecido, a função a estima a partir do fator de visão do céu da LCZ quando `lc_z` é dado (`Tmrt ≈ Ta + (1 − SVF) × 15`, uma aproximação simplificada da retenção de onda longa em cânions urbanos), ou simplesmente assume `Tmrt = air_temp` se nenhum dos dois for fornecido — portanto, passar `lc_z` é o que realmente torna o UTCI "consciente da LCZ", em vez de um índice meteorológico genérico.

**Nota sobre este exemplo**: o lcz4r_python atualmente não inclui nenhum conjunto de dados de estação meteorológica com velocidade do vento e umidade relativa (o CSV de Berlim usado em outros notebooks da série `local/` só tem temperatura do ar), então o exemplo abaixo é **ilustrativo/sintético**: aplicamos uma leitura meteorológica plausível de tarde de onda de calor uniformemente sobre um pequeno transecto de classes LCZ, para isolar e mostrar o efeito da forma urbana (via SVF) sobre o estresse térmico quando o tempo real é mantido constante.

In [5]:
from LCZ4py.local.lcz_thermal import lcz_utci
from LCZ4py._internal.lcz_parameters_data import LCZ_NAMES

# Illustrative/synthetic example -- NOT real station data.
# lcz4r_python ships no bundled meteorological record with wind speed and
# relative humidity (the Berlin CSV used elsewhere in local/ only has airT),
# so we construct a small transect of LCZ classes and apply ONE uniform,
# plausible heatwave-afternoon weather reading to all of them -- as if a
# single weather station reported this for the whole city.
lcz_transect = np.array([1, 2, 6, 9, 14, 17])  # compact core -> open -> natural -> water
transect_names = [LCZ_NAMES[c - 1] for c in lcz_transect]

air_temp = 34.0          # degC, typical heatwave afternoon
wind_speed = 2.0         # m/s, light breeze
relative_humidity = 45.0 # %

print(list(zip(lcz_transect.tolist(), transect_names)))

[(1, 'Compact highrise'), (2, 'Compact midrise'), (6, 'Open lowrise'), (9, 'Sparsely built'), (14, 'Low plants'), (17, 'Water')]


### `output="index"` — valores contínuos de UTCI

In [6]:
utci_index = lcz_utci(
    air_temp, wind_speed, relative_humidity,
    lc_z=lcz_transect, output="index",
)

for name, val in zip(transect_names, utci_index.values):
    print(f"  {name:<20s}  UTCI = {val:5.1f} degC")

print("\nSummary stats:", utci_index.stats)
utci_index.plot

  Compact highrise      UTCI =  38.6 degC
  Compact midrise       UTCI =  37.4 degC
  Open lowrise          UTCI =  35.0 degC
  Sparsely built        UTCI =  34.2 degC
  Low plants            UTCI =  33.8 degC
  Water                 UTCI =  33.8 degC

Summary stats: {'mean': 35.449852284648934, 'min': 33.76035245481622, 'max': 38.64015226348531, 'std': 1.8978867601236322, 'category_percentages': {'Extreme cold stress': 0.0, 'Very strong cold stress': 0.0, 'Strong cold stress': 0.0, 'Moderate cold stress': 0.0, 'Slight cold stress': 0.0, 'No thermal stress': 0.0, 'Moderate heat stress': 0.0, 'Strong heat stress': 83.33333333333334, 'Very strong heat stress': 16.666666666666664, 'Extreme heat stress': 0.0}}


**Lendo a saída do índice**: cada um dos seis pontos do transecto recebeu exatamente a *mesma* temperatura do ar, velocidade do vento e umidade — a única coisa que difere é a classe LCZ e, portanto, seu fator de visão do céu. Compact Highrise (LCZ 1) e Compact Midrise (LCZ 2) — as classes de menor SVF e mais fechadas do transecto — devem apresentar o UTCI mais alto, pois sua Tmrt estimada é elevada bem acima da temperatura do ar pela retenção de onda longa. Água (LCZ 17) e Low Plants (LCZ 14) — classes abertas, de alto SVF — devem ficar mais próximas da temperatura do ar. Esse é o resultado prático do UTCI consciente da LCZ: a *mesma* leitura de uma estação meteorológica se traduz em estresse térmico real muito diferente dependendo de onde na cidade uma pessoa está.

### `output="category"` — categorias de estresse térmico

As categorias vêm diretamente da tabela `UTCI_CATEGORIES` em `lcz_thermal.py` — dez faixas, de *Extreme cold stress* (estresse extremo por frio) até *No thermal stress* (sem estresse térmico) e *Extreme heat stress* (estresse extremo por calor), seguindo a escala padrão de avaliação do UTCI usada em orientações de saúde relacionadas ao calor.

In [7]:
utci_category = lcz_utci(
    air_temp, wind_speed, relative_humidity,
    lc_z=lcz_transect, output="category",
)

for name, cat in zip(transect_names, utci_category.categories):
    print(f"  {name:<20s}  -> {cat}")

  Compact highrise      -> Very strong heat stress
  Compact midrise       -> Strong heat stress
  Open lowrise          -> Strong heat stress
  Sparsely built        -> Strong heat stress
  Low plants            -> Strong heat stress
  Water                 -> Strong heat stress


**Lendo as categorias**: sob essa forçante sintética de tarde de onda de calor, espera-se que as classes construídas de baixo SVF (LCZ 1, 2) caiam em uma categoria de estresse térmico mais forte do que as classes abertas e naturais (LCZ 9, 14, 17) — mesmo que cada ponto tenha recebido entrada idêntica de temperatura do ar, vento e umidade. Esse é exatamente o tipo de sinal diferenciado de saúde relacionada ao calor, em nível de bairro, que uma única leitura de estação meteorológica da cidade não consegue fornecer sozinha, e que motiva combinar o UTCI com um mapa LCZ em aplicações de alerta precoce de saúde relacionada ao calor.

## Conclusão

Este notebook cobriu as duas funções térmicas puramente baseadas em arrays do `lcz4r_python`: `lcz_utci`, que transforma temperatura do ar, vento, umidade e uma temperatura radiante média derivada da LCZ em um índice e categoria padronizados de estresse térmico humano; e `lcz_anthropogenic_heat`, que estima o fluxo de calor residual (Q_F) que as próprias LCZs urbanas densas devolvem ao balanço de energia local, seja por uma consulta rápida baseada na literatura (`"simple"`) ou por uma parametrização orientada pela morfologia (`"detailed"`). Junto com as análises de superfície/dossel dos notebooks anteriores, essas ferramentas conectam a classificação LCZ diretamente a quantidades acionáveis de saúde humana e balanço de energia.

**Anterior**: [`06_temporal_climate_metrics`](06_temporal_climate_metrics.pt.ipynb) — amplitude térmica diurna e graus-hora por classe LCZ.

**Próximo**: [`08_drought_indices_spi_spei`](08_drought_indices_spi_spei.pt.ipynb) — índices de seca baseados em precipitação (SPI/SPEI) calculados por município.